In [189]:
# imports
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, classification_report

In [190]:
# load data
train_df_raw = pd.read_csv("../data/raw/train.csv")
test_df_raw = pd.read_csv("../data/raw/test.csv")

In [191]:
# clean data
#make it so I can run this cell over and over again
train_df = train_df_raw
test_df = test_df_raw


## encode sex
sex_map = {
    "male":0,
    "female":1
}
train_df["Sex"] = train_df["Sex"].replace(sex_map)

#encode Embarked
train_df = pd.get_dummies(train_df, columns=["Embarked"], dtype = int)
train_df

#drop cabin number and ticket number
train_df = train_df.drop(columns=["Cabin", "Ticket"])

# fill fare/age with median
train_df["Age"] = train_df["Age"].fillna(train_df["Age"].median())
train_df["Fare"] = train_df["Fare"].fillna(train_df["Fare"].median())

#save before I drop name
train_df_title = train_df.copy()

#drop name cause its a string
train_df = train_df.drop(columns=["Name"])

#save ids
train_ids = train_df["PassengerId"]
test_ids = test_df["PassengerId"]

# drop ids
train_df = train_df.drop(columns=["PassengerId"])
test_df = test_df.drop(columns=["PassengerId"])

# test set changes
test_df["Sex"] = test_df["Sex"].replace(sex_map)
test_df = pd.get_dummies(test_df, columns=["Embarked"], dtype = int)
test_df = test_df.drop(columns=["Cabin", "Ticket"])
test_df["Age"] = test_df["Age"].fillna(test_df["Age"].median())
test_df["Fare"] = test_df["Fare"].fillna(test_df["Fare"].median())
test_df = test_df.drop(columns=["Name"])

train_df

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S
0,0,3,0,22.0,1,0,7.2500,0,0,1
1,1,1,1,38.0,1,0,71.2833,1,0,0
2,1,3,1,26.0,0,0,7.9250,0,0,1
3,1,1,1,35.0,1,0,53.1000,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,0,1
...,...,...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,0,0,1
887,1,1,1,19.0,0,0,30.0000,0,0,1
888,0,3,1,28.0,1,2,23.4500,0,0,1
889,1,1,0,26.0,0,0,30.0000,1,0,0


In [192]:
#create my model
X = train_df.drop(columns = ["Survived"])
y = train_df["Survived"]

#split
X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

#train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

# get predictions
predictions = model.predict(X_val)

In [193]:
#inspect model
accuracy = accuracy_score(y_val, predictions)
print(accuracy)

print(confusion_matrix(y_val, predictions))
print(classification_report(y_val, predictions))

0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [194]:
# combine family columns and add one for the oerson themself. Feature engineering
train_df_fam = train_df.copy()
train_df_fam["family_size"] = train_df_fam["SibSp"] + train_df_fam["Parch"] + 1
# train_df_fam = train_df_fam.drop(columns=["SibSp", "Parch"])
train_df_fam

#create my model
X_fam = train_df_fam.drop(columns = ["Survived"])
y_fam = train_df_fam["Survived"]

#split
X_train_fam, X_val_fam, y_train_fam, y_val_fam = train_test_split(
    X_fam,
    y_fam,
    test_size=0.2,
    random_state=42
)

#train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_fam, y_train_fam)

# get predictions
fam_predictions = model.predict(X_val_fam)

In [195]:
#inspect fam model
accuracy_fam = accuracy_score(y_val_fam, fam_predictions)
print(accuracy_fam)

print(confusion_matrix(y_val_fam, fam_predictions))
print(classification_report(y_val_fam, fam_predictions))

0.8100558659217877
[[90 15]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.86      0.84       105
           1       0.79      0.74      0.76        74

    accuracy                           0.81       179
   macro avg       0.81      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179



In [196]:
# title extraction
train_df_title["Title"] = train_df_title["Name"].str.extract(r",\s*([^\.]+)\.")
train_df_title = train_df_title.drop(columns=["PassengerId", "Name"])

#title grouping
misc_titles = [
    "Major",
    "Mlle",
    "Col",
    "Don",
    "Mme",
    "Ms",
    "Lady",
    "Sir",
    "Capt",
    "the Countess",
    "Jonkheer",
    "Rev",
    "Dr"
]
train_df_title["Title"] = train_df_title["Title"].replace(misc_titles, 'Misc')

print(train_df_title["Title"].value_counts())

#encode titles
train_df_title = pd.get_dummies(train_df_title, columns=["Title"], dtype = int)

train_df_title

Title
Mr        517
Miss      182
Mrs       125
Master     40
Misc       27
Name: count, dtype: int64


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked_C,Embarked_Q,Embarked_S,Title_Master,Title_Misc,Title_Miss,Title_Mr,Title_Mrs
0,0,3,0,22.0,1,0,7.2500,0,0,1,0,0,0,1,0
1,1,1,1,38.0,1,0,71.2833,1,0,0,0,0,0,0,1
2,1,3,1,26.0,0,0,7.9250,0,0,1,0,0,1,0,0
3,1,1,1,35.0,1,0,53.1000,0,0,1,0,0,0,0,1
4,0,3,0,35.0,0,0,8.0500,0,0,1,0,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,0,27.0,0,0,13.0000,0,0,1,0,1,0,0,0
887,1,1,1,19.0,0,0,30.0000,0,0,1,0,0,1,0,0
888,0,3,1,28.0,1,2,23.4500,0,0,1,0,0,1,0,0
889,1,1,0,26.0,0,0,30.0000,1,0,0,0,0,0,1,0


In [197]:
#create my model
X_title = train_df_title.drop(columns = ["Survived"])
y_title = train_df_title["Survived"]

#split
X_train_title, X_val_title, y_train_title, y_val_title = train_test_split(
    X_title,
    y_title,
    test_size=0.2,
    random_state=42
)

#train model
model_title = LogisticRegression(max_iter=1000)
model_title.fit(X_train_title, y_train_title)

# get predictions
title_predictions = model_title.predict(X_val_title)

In [198]:
#inspect title model
accuracy_title = accuracy_score(y_val_title, title_predictions)
print(accuracy_title)

print(confusion_matrix(y_val_title, title_predictions))
print(classification_report(y_val_title, title_predictions))

0.8100558659217877
[[88 17]
 [17 57]]
              precision    recall  f1-score   support

           0       0.84      0.84      0.84       105
           1       0.77      0.77      0.77        74

    accuracy                           0.81       179
   macro avg       0.80      0.80      0.80       179
weighted avg       0.81      0.81      0.81       179

